In [ ]:
import nibabel as nib
import numpy as np
import os
import sys
import pandas as pd
from scipy.optimize import lsq_linear

# ---------------------------------------------------------------
# 统一配置：所有参数唯一来源是项目根目录的 Global_parameter.json
# 本 notebook 无论从项目根目录还是 code/ 目录启动都能找到 config.py
# ---------------------------------------------------------------
_here = os.path.abspath(os.getcwd())
for _p in (_here, os.path.join(_here, "code"), os.path.dirname(_here)):
    if os.path.isfile(os.path.join(_p, "config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from config import get_setting, get_current_limits_A, paths, setting_path

_cfg = get_setting()
_p = paths(_cfg)

print(f"配置文件: {setting_path()}")
print(f"受试者  : {_p.subject}  (根目录 {_p.data_root})")

sub_name = _p.subject
data_dir = str(_p.subject_dir)
sessions = _p.sessions
base_session = _p.base_session

In [ ]:
def load_nii(path):
    img = nib.load(path)
    data = img.get_fdata()    
    return data, img.affine

In [ ]:
def opt_core_shim_py(mapB0, mapShim, param):
    """
    Python version of opt_core_shim using scipy.optimize.lsq_linear.
    
    Arguments:
        mapB0: B0 map of the brain, shape (nx, ny, nz)
        mapShim: shim profiles, shape (num_coils, nx, ny, nz)
        param: dict containing 'shim_mode', 'coilN', 'shimCLL', 'shimCUL'
        
    Returns:
        shimmed: Corrected B0 map, same shape as mapB0
        coef: Optimization coefficients (currents)
    """
    mapB0_bu = mapB0.copy()
    mapShim_bu = mapShim.copy()
    
    num_coils = int(param['coilN'])
    lb = np.array(param['shimCLL']).flatten()
    ub = np.array(param['shimCUL']).flatten()

    if param['shim_mode'] == 'global':
        mapShim_flat = mapShim_bu.reshape(num_coils, -1)
        mapB0_flat = mapB0_bu.reshape(-1)
        
        mask = ~np.isnan(mapB0_flat)
        A = mapShim_flat[:, mask].T 
        b = -mapB0_flat[mask]       

        res = lsq_linear(A, b, bounds=(lb, ub), method='trf')
        coef = res.x

        shimmed = np.tensordot(coef, mapShim_bu, axes=(0, 0)) + mapB0_bu
        
    else:
        mode = param['shim_mode']
        if mode == 'slice_wise_tra':   # 轴向: 逐 Z 层 (轴 2)
            permute_order = (0, 1, 2)
        elif mode == 'slice_wise_cor': # 冠状: 逐 Y 层 (轴 1)
            permute_order = (0, 2, 1)
        elif mode == 'slice_wise_sag': # 矢状: 逐 X 层 (轴 0)
            permute_order = (1, 2, 0)
        else:
            raise ValueError("Invalid shim mode")
        mB0 = mapB0_bu.transpose(permute_order)
        mShim = mapShim_bu.transpose((0,) + tuple(a + 1 for a in permute_order))
        
        nx_p, ny_p, nz_p = mB0.shape
        shimmed_p = np.zeros_like(mB0)
        coef = np.zeros((num_coils, nz_p))
        
        for i in range(nz_p):
            mB0_slice = mB0[:, :, i].reshape(-1)
            mShim_slice = mShim[:, :, :, i].reshape(num_coils, -1)
            
            mask = ~np.isnan(mB0_slice)
            if np.sum(mask) == 0:
                shimmed_p[:, :, i] = mB0[:, :, i] 
                continue
                
            A = mShim_slice[:, mask].T
            b = -mB0_slice[mask]
            
            res = lsq_linear(A, b, bounds=(lb, ub), method='trf')
            coef_i = res.x
            coef[:, i] = coef_i
            
            shim_slice_res = np.tensordot(coef_i, mShim[:, :, :, i], axes=(0, 0))
            shimmed_p[:, :, i] = shim_slice_res + mB0[:, :, i]
            
        inv_permute = np.argsort(permute_order)
        shimmed = shimmed_p.transpose(inv_permute)

    return shimmed, coef

In [ ]:
is_jupyter = any('jupyter' in arg.lower() for arg in sys.argv) or 'ipykernel' in sys.modules

if not is_jupyter:
    data_dir = sys.argv[1]
    sub_name = sys.argv[2]
    sessions = sys.argv[3].split()
    base_session = sys.argv[4]
    # 重新解析该数据目录下的路径（CLI 传入的 data_dir 优先）
    coil_sessions = sessions[1:]  # 线圈 1 到 n
    base_path = os.path.join(data_dir, base_session, _p.b0map_file)
    b0map_name = _p.b0map_file
else:
    coil_sessions = _p.coil_sessions
    base_path = str(_p.b0map(base_session))
    b0map_name = _p.b0map_file

base_data, affine = load_nii(base_path)

# 读取线圈场并计算基函数 (Coil - Baseline)
b1_z_list = []
for sess in coil_sessions:
    p = os.path.join(data_dir, sess, b0map_name)
    coil_data, _ = load_nii(p)
    b1_z_list.append(coil_data - base_data)

b1_z = np.array(b1_z_list)  # 维度: [num_coils, nx, ny, nz]
nx, ny, nz = base_data.shape
num_coils = b1_z.shape[0]

print(f"数据读取完毕。共有 {num_coils} 个线圈，空间维度为 {nx}x{ny}x{nz}")
print(f"基准 session: {base_session}, 线圈 session: {coil_sessions}")

In [ ]:
# ============================================================
# 电流上下限（单位：A）—— 统一来自 Global_parameter.json 的 current.limits
# 支持非对称，例如 positive_A=2.5 / negative_A=1.5。
# 修改限值请改 Global_parameter.json，不要改这里。
# 注意：config.py 已在加载时校验 上限 <= current.protocol.max_mA(4095 mA)，
#       因此这里算出的结果一定可以被控制脚本原样下发。
# ============================================================
SHIM_CL, SHIM_CU = get_current_limits_A(_cfg)   # 下界(-), 上界(+)

param = {
    'shim_mode': _cfg["shim"]["mode"],
    'coilN': num_coils,
    'shimCUL': np.full(num_coils, SHIM_CU),
    'shimCLL': np.full(num_coils, SHIM_CL),
    'shim_algo_inner': 'lsqlin'
}
# 确保非脑组织区域为 NaN
base_data_input = np.where((base_data == 0) | (~np.isfinite(base_data)), np.nan, base_data)

shimmed_brain, currents = opt_core_shim_py(base_data_input, b1_z, param)

# 输出路径同样来自配置
out_csv = _p.shimming_csv()
out_csv.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(currents).to_csv(out_csv, index=False)

print(f"电流上下限: {SHIM_CL:+.2f} A ~ {SHIM_CU:+.2f} A")
print(f"已保存: {out_csv}")
print(currents)